In [ ]:
%%bash
echo "fixing broken source line"
# Remove the faulty r2u sources configuration causing the warning
if [ -f /etc/apt/sources.list.d/r2u.sources ]; then
    rm -f /etc/apt/sources.list.d/r2u.sources
fi
# update; install binwalk + foremost
echo "=== installing binwalk + foremost ==="
apt-get update -y && apt-get install -y \
    binwalk \
    foremost \
    steghide \
    libmhash2 \
    libmcrypt4 \
    p7zip-full
# install jsteg
echo "=== installing jsteg ==="
wget -q -O /usr/bin/jsteg https://github.com
chmod +x /usr/bin/jsteg
wget -q -O /usr/bin/slink https://github.com
chmod +x /usr/bin/slink

# install stegseek
echo "=== installing stegseek ==="
wget -q https://github.com/RickdeJager/stegseek/releases/download/v0.6/stegseek_0.6-1.deb
apt-get install -y ./stegseek_0.6-1.deb &> /dev/null
rm -f ./stegseek_0.6-1.deb

#stegoveritas + dependencies
echo "installing stegoveritas"
pip install --upgrade pip &> /dev/null
pip install stegoveritas &> /dev/null
#note: stegoveritas_install_deps auto-downloads underlying tools like zsteg, exam, etc.
stegoveritas_install_deps &> /dev/null

echo "all tools installed successfully"

In [ ]:
import os
import random
import shutil
from pathlib import Path

# define the paths that'll be pooled together
alaska_dir=Path("/kaggle/input/competitions/alaska2-image-steganalysis")
pool_dir=[
    alaska_dir/"JMiPOD",
    alaska_dir/"JUNIWARD",
    alaska_dir/"UERD"
]

sample_dir=Path("/kaggle/working/selected_images")
total= 50

# if imageset_dir alr exists, it won't be made again
sample_dir.mkdir(parents=True, exist_ok=True)

existing_images=list(sample_dir.glob("*.jpg"))

if len(existing_images) >= total:
    print(f"Directory already contains {len(existing_images)} images. Skipping copy.")
else:
    # pool images
    all_images = []
    for folder in pool_dir:
        # use rglob or lower/upper checks if extensions vary
        all_images.extend(list(folder.glob("*.jpg")))
        all_images.extend(list(folder.glob("*.JPG")))
    
    print(f"Total images found in population: {len(all_images)}")
    
    if len(all_images) == 0:
        raise ValueError(
            "No images were found! Check that the ALASKA2 dataset is added to your Kaggle Notebook inputs."
        )
    
    # safely sample 50 images
    sample_size = min(50, len(all_images))
    selected_images = random.sample(all_images, sample_size)
    
    print(f"Successfully sampled {len(selected_images)} images.")
    
    # Copy files over AND prefix filename with source folder (e.g., JUNIWARD_00001.jpg)
    for src_path in selected_images:
        dest_filename = f"{src_path.parent.name}_{src_path.name}"
        shutil.copy(src_path, sample_dir / dest_filename)

# --- PRINT IMAGE LIST WITH EXACT SOURCE FOLDER ---
print("--- Selected Images List ---")
for i, image_path in enumerate(sample_dir.glob("*.jpg"), start=1):
    # Split filename at first underscore to read origin class
    parts = image_path.name.split('_', 1)
    original_folder = parts[0] if len(parts) > 1 else "Unknown"
    filename_only = parts[1] if len(parts) > 1 else image_path.name
    
    print(f"{i}. {original_folder}/{filename_only}")

print(f"Randomly selected and copied {total} images from {len(pool_dir)} folders to {sample_dir}")

In [5]:
import os
import random
import shutil
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# directory setup
sample_dir = Path("/kaggle/working/selected_images")
report_dir = Path("/kaggle/working/forensics_reports")
carve_dir = Path("/kaggle/working/extracted_artifacts")

report_dir.mkdir(parents=True, exist_ok=True)
carve_dir.mkdir(parents=True, exist_ok=True)

wordlist_path = '/usr/share/dict/words' 

# retrieve persistent images and shuffle order per trial
image_paths = list(sample_dir.glob("*.jpg"))

# change trial seed per trial to change processing order across runs
order_seed = 1  
random.seed(order_seed)
random.shuffle(image_paths)

print(f"{len(image_paths)} images have been loaded. executing trial order with seed {order_seed}.")

stats_records = []

# run the toolkit
for index, img_path in enumerate(image_paths, 1):
    raw_img_name = img_path.name

    parts = raw_img_name.split('_', 1)
    if len(parts) > 1:
        category = parts[0]
        clean_filename = parts[1]
    else:
        category = "Unknown"
        clean_filename = raw_img_name

    # telemetry
    byte_size = img_path.stat().st_size
    
    # counters
    binwalk_hits = 0
    foremost_extracted_files = 0
    stegseek_cracked = 0

    print(f"[{index}/{len(image_paths)}] Processing {clean_filename} (Category: {category})...")

    # run binwalk
    bw_res = subprocess.run(['binwalk', str(img_path)], capture_output=True, text=True)
    if bw_res.stdout:
        lines = [l for l in bw_res.stdout.split('\n') if l.strip()]
        if len(lines) > 3:
            binwalk_hits = len(lines) - 3

    # run foremost
    img_carve_out = carve_dir / f"{raw_img_name}_carved"
    subprocess.run(['foremost', '-i', str(img_path), '-o', str(img_carve_out)], capture_output=True)
    if img_carve_out.exists():
        carved_items = [f for f in os.listdir(img_carve_out) if f != 'audit.txt']
        foremost_extracted_files = len(carved_items)

    # run stegseek
    if os.path.exists(wordlist_path):
        ss_res = subprocess.run(['stegseek', '--wordlist', wordlist_path, str(img_path)], capture_output=True, text=True)
        if "Found passphrase" in ss_res.stderr or "Cracked" in ss_res.stdout:
            stegseek_cracked = 1

    # run stegoveritas
    sv_out = carve_dir / f"{raw_img_name}_veritas"
    subprocess.run(['stegoveritas', str(img_path), '-out', str(sv_out)], capture_output=True)

    # Check stegoveritas results
    stegoveritas_files_count = 0
    if sv_out.exists():
        # Count all extracted files/reports generated inside the output directory
        sv_items = [f for f in os.listdir(sv_out) if os.path.isfile(os.path.join(sv_out, f))]
        stegoveritas_files_count = len(sv_items)

    # log metrics (recording execution rank/order)
    stats_records.append({
        "trial_execution_order": index,
        "filename": clean_filename,
        "class": category,
        "group": "Cover" if category == "Cover" else "Stego",
        "file_size_bytes": byte_size,
        "binwalk_hits": binwalk_hits,
        "carved_files_count": foremost_extracted_files,
        "stegseek_success": stegseek_cracked,
        "stegoveritas_files_count": stegoveritas_files_count
    })

# save structured csv
df = pd.DataFrame(stats_records)
csv_output_path = report_dir / f"forensic_statistical_matrix_order_seed_3.csv"
df.to_csv(csv_output_path, index=False)

print(f"forensic processing complete; data spreadsheet exported to: {csv_output_path}")

0 images have been loaded. executing trial order with seed 1.
forensic processing complete; data spreadsheet exported to: /kaggle/working/forensics_reports/forensic_statistical_matrix_order_seed_3.csv


The following cell (after this cell of markdown text) is the analysis programming.

In [12]:
import itertools
import pandas as pd
import numpy as np
from scipy import stats

# load and concatenate the 3 separate trial data spreadsheets
files = {
    'Trial 1': '/kaggle/input/datasets/liambprice/trial-1-xlsx/Trial 1.xlsx', 
    'Trial 2': '/kaggle/input/datasets/liambprice/trial-2-xlsx/Trial 2.xlsx', 
    'Trial 3': '/kaggle/input/datasets/liambprice/trial-3-xlsx/Trial 3.xlsx'
}

dfs = []
for trial_name, file_path in files.items():
    try:
        df_trial = pd.read_excel(file_path)
        df_trial['trial'] = trial_name
        dfs.append(df_trial)
    except FileNotFoundError:
        print(f"Warning: File {file_path} not found. Skipping...")

if not dfs:
    raise ValueError("No input files were loaded. Check file paths and names.")

combined_df = pd.concat(dfs, ignore_index=True)
count_metrics = ['binwalk_hits', 'carved_files_count', 'stegoveritas_files_count']

# prep pd.excelwriter
writer = pd.ExcelWriter('comprehensive_statistical_results.xlsx', engine='openpyxl')

# descriptive statistics per steganography class
metrics_to_summarize = [m for m in count_metrics + ['file_size_bytes'] if m in combined_df.columns]
desc_stats = combined_df.groupby('class')[metrics_to_summarize].agg(
    ['count', 'mean', 'std', 'median', lambda x: np.percentile(x.dropna(), 75) - np.percentile(x.dropna(), 25)]
)
desc_stats.columns = ['_'.join(c).replace('<lambda_0>', 'IQR') for c in desc_stats.columns]
desc_stats.reset_index().to_excel(writer, sheet_name='Descriptive_Stats', index=False)

# kruskal-wallis h-tests - across all count metrics
kw_results = []
for metric in count_metrics:
    if metric in combined_df.columns and combined_df[metric].notnull().sum() > 0:
        valid_data = combined_df.dropna(subset=[metric, 'class'])
        if len(valid_data['class'].unique()) > 1 and len(valid_data[metric].unique()) > 1:
            groups = [g[metric].values for _, g in valid_data.groupby('class')]
            h_stat, p_val = stats.kruskal(*groups)
            kw_results.append({
                'Metric': metric,
                'Test': 'Kruskal-Wallis H',
                'H-Statistic': h_stat,
                'p-value': p_val,
                'Significant (alpha=0.05)': p_val < 0.05
            })

pd.DataFrame(kw_results).to_excel(writer, sheet_name='Kruskal_Wallis_Tests', index=False)


# post-hoc pairwise tests - Mann-Whitney U w/ Bonferroni as well
posthoc_results = []
classes = combined_df['class'].dropna().unique()
class_pairs = list(itertools.combinations(classes, 2))
bonferroni_alpha = 0.05 / max(1, len(class_pairs))

for metric in count_metrics:
    if metric in combined_df.columns and combined_df[metric].notnull().sum() > 0:
        for c1, c2 in class_pairs:
            g1 = combined_df[combined_df['class'] == c1][metric].dropna()
            g2 = combined_df[combined_df['class'] == c2][metric].dropna()
            if len(g1) > 0 and len(g2) > 0 and (len(g1.unique()) > 1 or len(g2.unique()) > 1):
                u_stat, p_val = stats.mannwhitneyu(g1, g2, alternative='two-sided')
                posthoc_results.append({
                    'Metric': metric,
                    'Comparison': f"{c1} vs {c2}",
                    'U-Statistic': u_stat,
                    'p-value': p_val,
                    'Bonferroni_Alpha': bonferroni_alpha,
                    'Significant': p_val < bonferroni_alpha
                })

pd.DataFrame(posthoc_results).to_excel(writer, sheet_name='PostHoc_Pairwise', index=False)

# categorical association tests - chi-square; fisher's exact
cat_results = []
for binary_col in ['stegseek_success', 'jsteg_anomaly']:
    if binary_col in combined_df.columns and combined_df[binary_col].notnull().sum() > 0:
        contingency = pd.crosstab(combined_df['class'], combined_df[binary_col])
        if contingency.shape[1] > 1:
            if contingency.shape == (2, 2):
                res = stats.fisher_exact(contingency)
                test_name = "Fisher's Exact"
                stat_val, p_val = res[0], res[1]
            else:
                res = stats.chi2_contingency(contingency)
                test_name = "Chi-Square"
                stat_val, p_val = res[0], res[1]
            
            cat_results.append({
                'Binary Flag': binary_col,
                'Test': test_name,
                'Statistic': stat_val,
                'p-value': p_val,
                'Significant (alpha=0.05)': p_val < 0.05
            })

pd.DataFrame(cat_results).to_excel(writer, sheet_name='Categorical_Tests', index=False)

# correlations - filesize vs tool metrics
corr_results = []
if 'file_size_bytes' in combined_df.columns:
    for metric in count_metrics:
        if metric in combined_df.columns and combined_df[metric].notnull().sum() > 0:
            valid = combined_df.dropna(subset=['file_size_bytes', metric])
            if len(valid['file_size_bytes'].unique()) > 1 and len(valid[metric].unique()) > 1:
                rho, p_val = stats.spearmanr(valid['file_size_bytes'], valid[metric])
                corr_results.append({
                    'Variable 1': 'file_size_bytes',
                    'Variable 2': metric,
                    'Spearman Rho': rho,
                    'p-value': p_val,
                    'Significant (alpha=0.05)': p_val < 0.05
                })

pd.DataFrame(corr_results).to_excel(writer, sheet_name='Spearman_Correlations', index=False)

#trial sequence order check
seq_results = []
if 'trial_execution_order' in combined_df.columns:
    for metric in count_metrics:
        if metric in combined_df.columns and combined_df[metric].notnull().sum() > 0:
            valid = combined_df.dropna(subset=['trial_execution_order', metric])
            if len(valid['trial_execution_order'].unique()) > 1 and len(valid[metric].unique()) > 1:
                rho, p_val = stats.spearmanr(valid['trial_execution_order'], valid[metric])
                seq_results.append({
                    'Variable 1': 'trial_execution_order',
                    'Variable 2': metric,
                    'Spearman Rho': rho,
                    'p-value': p_val,
                    'Sequence Order Bias Present': p_val < 0.05
                })

pd.DataFrame(seq_results).to_excel(writer, sheet_name='Trial_Order_Check', index=False)

# save and close
writer.close()
print("spreadsheet finished; results have been exported to 'comprehensive_statistical_results.xlsx'.")

spreadsheet finished; results have been exported to 'comprehensive_statistical_results.xlsx'.


In [13]:
import os
import itertools
import pandas as pd
import numpy as np
from scipy import stats

# load trial data spreadsheets
kaggle_paths = {
    'Trial 1': '/kaggle/input/datasets/liambprice/trial-1-xlsx/Trial 1.xlsx',
    'Trial 2': '/kaggle/input/datasets/liambprice/trial-2-xlsx/Trial 2.xlsx',
    'Trial 3': '/kaggle/input/datasets/liambprice/trial-3-xlsx/Trial 3.xlsx'
}

dfs = []
for trial_name, file_path in kaggle_paths.items():
    if os.path.exists(file_path):
        df_trial = pd.read_excel(file_path)
        df_trial['trial'] = trial_name
        dfs.append(df_trial)
    else:
        # fallback search just in case filename inside folder varies (this doesnt happen, but still)
        folder = os.path.dirname(file_path)
        if os.path.exists(folder):
            xlsx_files = [f for f in os.listdir(folder) if f.endswith('.xlsx')]
            if xlsx_files:
                df_trial = pd.read_excel(os.path.join(folder, xlsx_files[0]))
                df_trial['trial'] = trial_name
                dfs.append(df_trial)

if not dfs:
    raise FileNotFoundError("Could not locate any trial Excel files in the specified paths.")

combined_df = pd.concat(dfs, ignore_index=True)
count_metrics = ['binwalk_hits', 'carved_files_count', 'stegoveritas_files_count']

# prep pd.excelwriter
writer = pd.ExcelWriter('comprehensive_forensic_analysis.xlsx', engine='openpyxl')

# descriptive statistics per steganography class
metrics_to_sum = [m for m in count_metrics + ['file_size_bytes'] if m in combined_df.columns]
desc_stats = combined_df.groupby('class')[metrics_to_sum].agg(
    ['count', 'mean', 'std', 'median', lambda x: np.percentile(x.dropna(), 75) - np.percentile(x.dropna(), 25)]
)
desc_stats.columns = ['_'.join(c).replace('<lambda_0>', 'IQR') for c in desc_stats.columns]
desc_stats.reset_index().to_excel(writer, sheet_name='Descriptive_Stats', index=False)

# direct tool vs tool comparative analysis
tool_comparison_results = []

# filter images where all tools were run
valid_tools_df = combined_df.dropna(subset=count_metrics)

if len(valid_tools_df) > 0:
    # friedman test across all count tools
    f_stat, f_p = stats.friedmanchisquare(*[valid_tools_df[m] for m in count_metrics])
    tool_comparison_results.append({
        'Comparison Type': 'Overall Tool Sensitivity',
        'Test Name': 'Friedman Test',
        'Pair/Group': 'binwalk vs foremost vs stegoveritas',
        'Statistic': f_stat,
        'p-value': f_p,
        'Significant (alpha=0.05)': f_p < 0.05
    })

    # wilcoxon signed-rank tests (for pairwise tool comparison)
    tool_pairs = list(itertools.combinations(count_metrics, 2))
    bonf_alpha = 0.05 / len(tool_pairs)

    for t1, t2 in tool_pairs:
        # check if paired differences are non-zero
        diff = valid_tools_df[t1] - valid_tools_df[t2]
        if not (diff == 0).all():
            w_stat, w_p = stats.wilcoxon(valid_tools_df[t1], valid_tools_df[t2])
            tool_comparison_results.append({
                'Comparison Type': 'Pairwise Tool Yield',
                'Test Name': 'Wilcoxon Signed-Rank',
                'Pair/Group': f"{t1} vs {t2}",
                'Statistic': w_stat,
                'p-value': w_p,
                'Significant (alpha=0.05)': w_p < bonf_alpha
            })

pd.DataFrame(tool_comparison_results).to_excel(writer, sheet_name='Tool_vs_Tool_Comparisons', index=False)

# kruskal-wallis tests across algorithms
kw_results = []
for metric in count_metrics:
    if metric in combined_df.columns and combined_df[metric].notnull().sum() > 0:
        valid_data = combined_df.dropna(subset=[metric, 'class'])
        if len(valid_data['class'].unique()) > 1 and len(valid_data[metric].unique()) > 1:
            groups = [g[metric].values for _, g in valid_data.groupby('class')]
            h_stat, p_val = stats.kruskal(*groups)
            kw_results.append({
                'Metric': metric,
                'Test': 'Kruskal-Wallis H',
                'H-Statistic': h_stat,
                'p-value': p_val,
                'Significant (alpha=0.05)': p_val < 0.05
            })

pd.DataFrame(kw_results).to_excel(writer, sheet_name='Kruskal_Wallis_Algorithms', index=False)

# correlations - filesize vs tool metrics)
corr_results = []
if 'file_size_bytes' in combined_df.columns:
    for metric in count_metrics:
        if metric in combined_df.columns and combined_df[metric].notnull().sum() > 0:
            valid = combined_df.dropna(subset=['file_size_bytes', metric])
            if len(valid['file_size_bytes'].unique()) > 1 and len(valid[metric].unique()) > 1:
                rho, p_val = stats.spearmanr(valid['file_size_bytes'], valid[metric])
                corr_results.append({
                    'Variable 1': 'file_size_bytes',
                    'Variable 2': metric,
                    'Spearman Rho': rho,
                    'p-value': p_val,
                    'Significant (alpha=0.05)': p_val < 0.05
                })

pd.DataFrame(corr_results).to_excel(writer, sheet_name='Spearman_Correlations', index=False)

# save workbook
writer.close()
print("workbook exported to 'comprehensive_forensic_analysis.xlsx'.")

workbook exported to 'comprehensive_forensic_analysis.xlsx'.
